# Generate Input files for RAVEN

## Contents
- [Input Datasets for this Notebook](#input-datasets-for-this-notebook)
- [0. Import required modules](#0-import-required-modules)
- [1. Organize data folders and files](#1-organize-data-folders-and-files)
    - [1.1 Create "Raven" subfolder](#11-create-raven-subfolder)
    - [1.2 Create subfolders in "Raven" subfolder ("Raw", "Interim", "Processed")](#12-create-subfolders-in-raven-subfolder-raw-interim-processed)
- [2. Create Raven input files](#2-create-raven-input-files)
    - [2.1 Import and project CLRH depression depths file ](#21-import-and-project-clrh-depression-depths-file)
    - [2.2 Calculations](#22-calculations)
    - [2.3 Create RVT File](#23-create-rvt-file)
    - [2.4 Create RVI File](#24-create-rvi-file)
    - [2.5 Create RVC File](#25-create-rvc-file)
    - [2.6 Create RVP File](#26-create-rvp-file)
    - [2.7 Create RVT File](#27-create-rvt-file)
    - [2.8 Create RVH File](#28-create-rvh-file)
    - [2.9 Create channel properties RVP file](#29-create-channel-properties-rvp-file)
    - [2.10 Create GridWeights TXT file](#210-create-gridweights-txt-file)

    
    
    




## Input Datasets for this Notebook

Prior to running this notebook, ```depression_depths_calc.ipynb``` and ```netCDF_add_dimension.ipynb``` scripts, in this order, must be ran first. In addition, the following files must be downloaded and pasted into the "Data" folder at the root project directory: 
- [WSC historic discharge data](https://wateroffice.ec.gc.ca/search/historical_e.html): search by outlet station name or number and download Daily data in Date-Data Format (CSV with missing days) (e.g., station number is 05OE006 for Manning)
- [WSC metadata](https://wateroffice.ec.gc.ca/station_metadata/station_characteristics_e.html) of desired outlet point.

Once the Raven input files are generated, the Raven model can be run. The [Raven executable file with NetCDF support](http://raven.uwaterloo.ca/Downloads.html) must be downloaded, and a copy of Raven.exe must be pasted into the folder containing the input files (Data/ModelName/Raven/Processed). For Windows users, the [NetCDF 4 Library (without DAP)](https://www.unidata.ucar.edu/software/netcdf/docs/winbin.html) must also be installed, and the directory path of the installed NetCDF.dll file must be in your PATH environment variable. For MacOS and linux users, instructions for installing NetCDF can be found in section 2.1 of the [Raven Manual](RavenManual_v3.8.pdf). Once Raven and NetCDF are installed in the correct locations, create a sub-folder within the Data/ModelName/Raven/Processed directory called “input”, and move the GridWeigths.txt file and the runoff netcdf file to this folder. Next, open the command prompt and navigate to your Data/Manning/Raven/Processed directory (in Windows, this can be done with “cd c:/Data/ModelName/Raven/Processed”). Finally, run the following command: “Raven.exe ModelName -o ./output/”, where ModelName is the name of your model, formatted the same as in the names of your Raven input files. The model outputs will be stored in a folder called “output” within the Data/ModelName/Raven/Processed directory. 

## 0. Import required modules

In [ ]:
# System
import os
import sys
import fileinput
from pathlib import Path
import shutil


# Analysis
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint


# Time
from datetime import datetime

## 1. Organize data folders and files

### 1.1 Create "Raven" subfolder

In [39]:
# Set watershed name
WATERSHED_NAME = r"Manning"


# Path for this notebook
NOTEBOOK_PATH = Path.cwd()


# Path for PWA root
PWA_PATH = str(NOTEBOOK_PATH.parent.as_posix())


# Path for base raw data folder
BS_DATA_PATH = PWA_PATH + r"/Data/"


# Path for watershed folder (created in depression_depths_calc.ipynb)
WATERSHED_PATH = BS_DATA_PATH + WATERSHED_NAME


# Path for Raven folder
RAVEN_PATH = WATERSHED_PATH + r"/Raven"


# Create Raven folder in specified path
os.makedirs(RAVEN_PATH, 
            exist_ok=True) # Do nothing if already exists

### 1.2 Create subfolders in "Raven" subfolder ("Raw", "Interim", "Processed")

In [40]:
# Specify paths for watershed subfolders
RA_RAW_PATH = RAVEN_PATH + r"/Raw/"
RA_INTERIM_PATH = RAVEN_PATH + r"/Interim/"
RA_PROCESSED_PATH = RAVEN_PATH + r"/Processed/"


# Create watershed subfolders in specified paths
SUBFOLDERS_LIST = [RA_RAW_PATH,
                   RA_INTERIM_PATH,
                   RA_PROCESSED_PATH]
for sub in SUBFOLDERS_LIST:
    os.makedirs(sub, exist_ok=True)

### 1.3 Move input data files to "Raw" subfolder

In [41]:
# Specify source and destination folders before moving files
src = Path(BS_DATA_PATH)
dst = Path(RA_RAW_PATH)


# Move files in base data folder to watershed folder
for file in src.iterdir():
     if file.is_file():
        destination_file = dst / file.name
        if destination_file.exists():
            print(f"Skipped (already exists): {file.name}")
            continue
        shutil.move(str(file), destination_file)

In [42]:
# Depression depths shapefile name (generated in depression_depths_calc.ipynb)
DEPRESSION_DEPTHS_SHAPEFILE = "CLRH_basins_depression_depths"


# Depression depths shapefile path
DEPRESSION_DEPTHS_PATH = WATERSHED_PATH + r"/DepressionDepths/Processed"


# Copy and paste depression depths files to "Raw" folder
for filename in os.listdir(DEPRESSION_DEPTHS_PATH):
    if filename.startswith(DEPRESSION_DEPTHS_SHAPEFILE):
        source_path = os.path.join(DEPRESSION_DEPTHS_PATH, filename)
        destination_path = os.path.join(RA_RAW_PATH, filename)
        if os.path.isfile(source_path):  # Avoid copying directories
            shutil.copy2(source_path, destination_path)  # copy2 preserves metadata

In [43]:
# PHyDAP file name (generated in netCDF_to_PHyDAP.ipynb)
PHYDAP_FILE_NAME = "era5_runoff_wlat.nc"


# PHyDAP file path
PHYDAP_FILE_PATH = WATERSHED_PATH + r"/Runoff/Processed"


# Copy and paste PHyDAP file to "Raw" folder
for filename in os.listdir(PHYDAP_FILE_PATH):
    if filename.startswith(PHYDAP_FILE_NAME):
        source_path = os.path.join(PHYDAP_FILE_PATH, filename)
        destination_path = os.path.join(RA_RAW_PATH, filename)
        if os.path.isfile(source_path):  # Avoid copying directories
            shutil.copy2(source_path, destination_path)  # copy2 preserves metadata

## 2. Create Raven input files

### 2.1 Import and project CLRH depression depths file 

In [44]:
# Read clrh shapefile
clrh_gdf = gpd.read_file(RA_RAW_PATH + 
                         f"/{DEPRESSION_DEPTHS_SHAPEFILE}.shp")


# Project clrh gdf to WGS 84
clrh_gdf_project = clrh_gdf.to_crs(4326)

### 2.2 Calculations

In [46]:
# CALCULATE TIME OF CONCENTRATION AND TIME TO PEAK


# Constants
CN = 63 # Curve number for Manning watershed
CN_RETENTION = (1000 / CN) - 10 # Retention parameter for Manning watershed
SQM_TO_ACRES = 0.00024710538 # Conversion factor from square meters to acres


# Area and slope conversion
clrh_gdf_project['BasinsArea_acres'] = clrh_gdf_project['BasArea'] * SQM_TO_ACRES
clrh_gdf_project['BasSlope_prcnt'] = np.tan(np.radians(clrh_gdf_project['BasSlope'])) * 100


# Flow length estimation
clrh_gdf_project['FlowLength_ft'] = 209 * clrh_gdf_project['BasinsArea_acres'] ** 0.6


# Time of concentration
flow_length = clrh_gdf_project['FlowLength_ft']
slope = clrh_gdf_project['BasSlope_prcnt']

clrh_gdf_project['max_retention'] = CN_RETENTION
clrh_gdf_project['time_of_conc'] = (flow_length ** 0.8 * (CN_RETENTION + 1) ** 0.7) / \
                                    (1140 * slope ** 0.5)


# Time to peak
clrh_gdf_project['time_to_peak'] = clrh_gdf_project['time_of_conc'] * 0.6

In [47]:
# Create lists of necessary columns
subids = list(clrh_gdf_project['SubId'])
rivlengths = list(clrh_gdf_project['RivLength'])
hasgauge = list(clrh_gdf_project['Has_POI'])
dowsubids = list(clrh_gdf_project['DowSubId'])
basarea = list(clrh_gdf_project['BasArea'])
elev = list(clrh_gdf_project['MeanElev'])
lat = list(clrh_gdf_project['centroid_y'])
lon = list(clrh_gdf_project['centroid_x'])
slope = list(clrh_gdf_project['BasSlope'])
aspect = list(clrh_gdf_project['BasAspect'])
bkfwidth = list(clrh_gdf_project['BkfWidth'])
bkfdepth = list(clrh_gdf_project['BkfDepth'])
rivslope = list(clrh_gdf_project['RivSlope'])
floodpn = list(clrh_gdf_project['FloodP_n'])
chn = list(clrh_gdf_project['Ch_n'])
timeofconc = list(clrh_gdf_project['time_of_conc'])
timetopeak = list(clrh_gdf_project['time_to_peak'])
depsdepth = list(clrh_gdf_project['Deps_Depth'])

In [ ]:
# MODIFY RIVER LENGTHS AND SLOPES


# Convert river lengths to km
rivlengths_km = [x / 1000 for x in rivlengths]


# Round to 3 decimals
rivlengths_km_rounded = [round(v, 3) for v in rivlengths_km]


# Replace invalid river lengths with 'ZERO-'
rivlengths_mod = list(map(lambda x: 'ZERO-' if x == -0.001 else x, \
                          rivlengths_km_rounded))


# Replace invalid river lengths with 0.00
rivlengths_mod_n = list(map(lambda x: 0.0 if x == -0.0012345 else x, \
                            rivlengths_km))


# Modify river slope
rivslope_mod = list(map(lambda x: 0.0001234 if x <= 0.0001 else x, \
                        rivslope))

<class 'float'> 1.472


In [50]:
# Convert basin areas to km2
basarea_km2 = [x / 1000000 for x in \
               basarea]

In [51]:
# Extract outlet subid and drainage area
outlet = clrh_gdf_project.loc[clrh_gdf_project['Has_POI'] == 1, 'SubId'].item()
outlet_da = clrh_gdf_project.loc[clrh_gdf_project['Has_POI'] == 1, 'DrainArea'].item()


# Convert drainage area to km2
outlet_da_km2 = outlet_da / 1000000

In [79]:
# Replace invalid bankfull depth values with 1
bkfdepth_mod = list(map(lambda x: 1 if x < 1 else x, \
                    bkfdepth))

In [80]:
# Replace invalid floodpn and chn values with 0.05 and 0.025, respectively
floodpn_mod = list(map(lambda x: 0.05 if x < 0 else x, \
                floodpn))
chn_mod = list(map(lambda x: 0.025 if x < 0 else x, \
               chn))

### 2.3 Create RVT File

In [56]:
# WSC metadata file name
WSC_METADATA_FILE_NAME = "metadata_20250603T1908"


# Read WSC metadata file
wsc_metadata_gdf = gpd.read_file(RA_RAW_PATH + WSC_METADATA_FILE_NAME + ".csv")


# WSC data file name
WSC_DATA_FILE_NAME = "daily_20250603T1908"


# Read WSC data file
wsc_data_df = pd.read_csv(RA_RAW_PATH + WSC_DATA_FILE_NAME + ".csv",
                          skiprows=1)


# Extract discharge values from dataframe and replace NaN values with -1.2345
wsc_data_dschrg = wsc_data_df.loc[wsc_data_df['PARAM']==1, 'Value']
wsc_data_dschrg.fillna(-1.2345, inplace=True)


# Extract start and end date from WSC dataframe
start_date = pd.to_datetime(wsc_data_df['Date'].iloc[0]).strftime('%Y-%m-%d')
end_date = pd.to_datetime(wsc_data_df['Date'].iloc[-1]).strftime('%Y-%m-%d')


# Create file and store in processed folder
FILE_PATH_RVT = RA_PROCESSED_PATH + "WSC" + \
            wsc_metadata_gdf['Station Number'].values[0] + ".rvt"


# Extract relevant info from WSC metadata file
station_name = wsc_metadata_gdf['Station Name'].iloc[0]
station_id = wsc_metadata_gdf['Station Number'].iloc[0]
num_points = len(wsc_data_dschrg)


# Write metadata to file
with open(FILE_PATH_RVT, 'w') as file:
    header = f"""\
#################################################################################
# Name = {station_name}
# ID = {station_id}
# Drainage Area = {int(outlet_da_km2)} km2
# Sub-basin ID = {outlet}
# Natural/Reference = Regulated
# Country = CA
# Date Range = {start_date} - {end_date}
# Data time-step = 24 hr
# Missing data are filled with -1.2345
##################################################################################
:ObservationData	HYDROGRAPH	{outlet}	m3/s
    {datetime.strptime(start_date, '%Y-%m-%d')}	1	{num_points}
"""
    file.write(header)


# Write data to file
with open(FILE_PATH_RVT, 'a') as file:
    for value in wsc_data_dschrg:
        file.write(f"\t{value:.4f}\n")
    file.write(':EndObservationData')


### 2.4 Create RVI File

In [57]:
# Create file and store in processed folder
FILE_PATH_RVI = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvi"


# Declare model run time
model_run_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')


# Write to file
with open(FILE_PATH_RVI, 'w') as file:
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------
:StartDate             {model_run_time:<30}# Model run start time, format in yyyy-mm-dd hh:mm:ss
:EndDate               {model_run_time:<30}# Model run end time, format in yyyy-mm-dd hh:mm:ss
:Method                ORDERED_SERIES                # Numerical method used for simulation
:TimeStep              01:00:00                      # Expressed in days as a real-valued number (e.g., 1.0 for one day) or using hh:mm:ss format (e.g., 24:00:00 for one hour).
:RunName               {WATERSHED_NAME:<30}# The name of model run. It will be added as a prefix of model output files.

:CatchmentRoute        ROUTE_TRI_CONVOLUTION         # Catchment routing method, used to convey water from the catchment tributaries and rivulets to the subbasin outlets. DEFAULT ROUTE_DUMP, which instantly ‘dumps’ all water in the subbasin stream reach.
:Routing               ROUTE_DIFFUSIVE_WAVE          # Channel routing method which is used to transport water from upstream to downstream within the main subbasin channels. DEFAULT ROUTE_DIFFUSIVE_WAVE, which analytically solves the diffusive wave equation along the reach using a constant reference celerity.
:PrecipIceptFract      PRECIP_ICEPT_NONE             # Estimation of the precipitation interception fraction. In this routing model, stream input(s) are "pretending" to be precipitation going into Raven, thus using DEFAULT PRECIP_ICEPT_NONE to indicate no interception processes are adopted.
:PotentialMeltMethod   POTMELT_NONE                  # Estimation of the potential snow melt. In this routing model, snow melt processes are not relevant, thus using DEFAULT POTMELT_NONE method.
:SWRadiationMethod     SW_RAD_NONE                   # Estimation of shortwave incoming radiation. This is turned off for a routing-only model.
:SoilModel             SOIL_ONE_LAYER                # In this routing model, use DEFAULT SOIL_ONE_LAYER to define single soil layer structure.

:HydrologicProcesses
  :Precipitation     PRECIP_RAVEN            ATMOS_PRECIP     PONDED_WATER          # Moves stream input(s) from ATMOS_PRECIP to PONDED_WATER storage (waiting for runoff). Use DEFAULT PRECIP_RAVEN method.
  :Abstraction       ABST_FILL               PONDED_WATER     DEPRESSION
  :Flush             RAVEN_DEFAULT           PONDED_WATER     SURFACE_WATER         # Moves water from PONDED_WATER to SURFACE_WATER (routed to outlet). Use DEFAULT RAVEN_DEFAULT method.
  :Seepage           SEEP_LINEAR             DEPRESSION       SOIL[0]
:EndHydrologicProcesses

# Defines the hydrograph performance metrics output by Raven. Either one or multiple is acceptable.
:EvaluationMetrics           NASH_SUTCLIFFE    PCT_BIAS    KLING_GUPTA
""")

### 2.5 Create RVC File

In [58]:
# Create file and store in processed folder
FILE_PATH_RVC = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvc"


# Write to file
with open(FILE_PATH_RVC, 'w') as file:
    # Write initial conditions to file
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------

:UniformInitialConditions SOIL[0] 15            # Initial conditions for the groundwater storage, in mm. DEFAULT 15 mm.
""")


### 2.6 Create RVP File

In [60]:
# Create file and store in processed folder
FILE_PATH_RVP = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvp"


# Write to file
with open(FILE_PATH_RVP, 'w') as file:
    # Write soil, vegetation, and land use content
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------

# Soil/Vegetation/Land use class definitions ------
:SoilClasses
  AQUIFER      # only one soil class used here
:EndSoilClasses

:SoilProfiles
  # name, number of layers, soil class, thickness [m]
  LandHRU_soil_profile,    1,    AQUIFER, 5
:EndSoilProfiles

# The vegetation and land use parameters unused in routing
:VegetationClasses
  :Attributes,                MAX_HT,       MAX_LAI,    MAX_LEAF_COND
  :Units,                       m,            none,       mm_per_s
  LandHRU_vege_class,          25,           5.0,          5.0
:EndVegetationClasses

:LandUseClasses
  :Attributes,        IMPERMEABLE_FRAC,         FOREST_COVERAGE
  :Units,                     fract,                    fract
  LandHRU_landuse_class,      0,                        1""")
    


# Append to file
with open(FILE_PATH_RVP, 'a') as file:
    # Append land use parameters for each subbasin
    for subid in subids:
        file.write(f"\n  SUB{subid},              0,                        1")

    file.write("""\
\n:EndLandUseClasses

# Global Parameters-----------------------------
:AvgAnnualRunoff        75

:LandUseParameterList
  :Parameters     ,          DEP_SEEP_K,             DEP_MAX
  :Units          ,               1/d,                   mm
    [DEFAULT]     ,             0.01,              20.0000""")

    # Write subbasin-specific parameter values
    for subid, depth in zip(subids, 
                            depsdepth):
        file.write(f"\n SUB{subid:<12},             0.01,          {depth:.4f}")

    file.write("""\

:EndLandUseClasses

# redirects to channel properties file
:RedirectToFile         channel_properties.rvp
""")

### 2.7 Create RVT File

In [61]:
# Create file and store in processed folder
FILE_PATH_RVT = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvt"


# Write to file
with open(FILE_PATH_RVT, 'w') as file:
    # Write PHyDAP netCDF metadata to file and point to GridWeights and WSC files
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------

:GriddedForcing           StreamInputs
    :ForcingType          PRECIP
    :FileNameNC           input/{PHYDAP_FILE_NAME}
    :VarNameNC            runoff
    :DimNamesNC           lon lat time     # must be in the order of (x,y,t)
    :LinearTransform      24.0     0.0
    :RedirectToFile       input/GridWeights.txt
:EndGriddedForcing

# NOTE: the Linear Transform converts mm/1hr --> mm/d

:GriddedForcing           Averagetemp
    :ForcingType          TEMP_AVE
    :FileNameNC           input/{PHYDAP_FILE_NAME}
    :VarNameNC            runoff
    :DimNamesNC           lon lat time
    :RedirectToFile       input/GridWeights.txt
:EndGriddedForcing

# Observed Discharge Data-----------------------
:RedirectToFile           WSC{wsc_metadata_gdf['Station Number'].values[0]}.rvt
""")

### 2.8 Create RVH File

In [ ]:
# Create file and store in processed folder
FILE_PATH_RVH = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvh"


# Write to file
with open(FILE_PATH_RVH, 'w') as file:
    # Write SubBasin metadata to file
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------

## SubBasin List---------------------------------------------------------------------
# ID:            A positive long integer identifier unique to this subbasin.
# NAME:          A nickname for subbasin. DEFAULT string("sub_" + ID).
# DOWNSTREAM_ID: The ID of the subbasin that receives this subbasin's outflowing waters.
# PROFILE:       The representative channel profile code (defined in channel_properties.rvp).
# REACH_LENGTH:  The length of the primary reach channel in the basin (in km).
# GAUGED:        Flag to indicate whether hydrographs for this subbasin are output (1 = yes, 0 = no).
#                In this routing model, both subbasins with reservoirs and flow gauges are GAUGED = 1.

:SubBasins
  :Attributes     ID     NAME     DOWNSTREAM_ID     PROFILE     REACH_LENGTH     GAUGED
  :Units          none   none     none              none        km              none
""")
    

# Append to file
with open(FILE_PATH_RVH, 'a') as file:
    # Append subbasin data to file
    for i in range(len(subids)):
        subid = subids[i]
        downstream_id = dowsubids[i]
        channel_profile = f"chn_{subid}"
        reach_length = rivlengths_mod[i]
        gauged = hasgauge[i]

        file.write(
            f"{subid:>10}  "                       # Subbasin ID
            f"{'sub' + str(subid):<15}  "          # Subbasin name
            f"{downstream_id:>10}  "               # Downstream Subbasin ID
            f"{channel_profile:<15}  "             # Channel profile name
            f"{reach_length:>10}  "             # Reach length (km)
            f"{gauged:>5}\n"                       # Gauged flag (1 or 0)
        )


# Append to file
with open(FILE_PATH_RVH, 'a') as file:
    # Append HRU metadata to file
    file.write(
        ":EndSubBasins\n\n"
        "## HRU List ---------------------------------------------------------------------\n"
        "# HRU ID:          A positive integer unique to this HRU. Note that more than one HRUs\n"
        "#                 may have the same subbasin ID, but HRU ID is unique.\n"
        "# AREA:            HRU area value, in km².\n"
        "# ELEVATION:       HRU mean elevation, in m.\n"
        "# LATITUDE:        Latitude of HRU centroid, in decimal degrees.\n"
        "# LONGITUDE:       Longitude of HRU centroid, in decimal degrees.\n"
        "# BASIN_ID:        ID of the subbasin in which the HRU is located.\n"
        "# LAND_USE_CLASS:  Land use class defined in .rvp file. DEFAULT LandHRU_landuse_class.\n"
        "# VEG_CLASS:       Vegetation class defined in .rvp file. DEFAULT LandHRU_vege_class.\n"
        "# SOIL_PROFILE:    Soil profile defined in .rvp file. DEFAULT LandHRU_soil_profile.\n"
        "# AQUIFER_PROFILE: Aquifer class. DEFAULT [NONE] in this routing model.\n"
        "# TERRAIN_CLASS:   Terrain class. DEFAULT [NONE] in this routing model.\n"
        "# SLOPE:           HRU mean slope, in degrees.\n"
        "# ASPECT:          HRU mean aspect, in degrees. North: 0°, West: 90°, South: 180°, East: 270°.\n"
        "#\n"
        ":HRUs\n"
        "  :Attributes     AREA     ELEVATION     LATITUDE     LONGITUDE     BASIN_ID     LAND_USE_CLASS         "
        "VEG_CLASS          SOIL_PROFILE     AQUIFER_PROFILE     TERRAIN_CLASS     SLOPE     ASPECT\n"
        "  :Units          km2      m             deg          deg           none         none                   "
        "none                none             none               none              deg       degN\n"
    )


# Append to file
with open(FILE_PATH_RVH, 'a') as file: 
    # Append HRU data to file
    for i in range(len(subids)):
        subid = subids[i]
        area = round(basarea_km2[i], 4)
        elevation = round(elev[i], 2)
        latitude = round(lat[i], 6)
        longitude = round(lon[i], 6)
        slope_val = round(slope[i], 2)
        aspect_val = round(aspect[i], 2)

        file.write(
            f"{subid:>15}"
            f"{area:>10}"
            f"{elevation:>10}"
            f"{latitude:>15}"
            f"{longitude:>15}"
            f"{subid:>15}"
            f"{'LandHRU_landuse_class':>30}"
            f"{'LandHRU_vege_class':>25}"
            f"{'LandHRU_soil_profile':>30}"
            f"{'[NONE]':>10}"
            f"{'[NONE]':>10}"
            f"{slope_val:>10}"
            f"{aspect_val:>10}\n"
        )

    file.write(':EndHRUs\n')


# Append to file
# Open file in append mode
with open(FILE_PATH_RVH, 'a') as file:
    # Append time of concentration and time to peak data to file
    file.write(
        '\n# SubBasin Properties ----------------------------------------------------\n'
        ':SubBasinProperties\n'
        '  :Parameters      TIME_CONC  TIME_TO_PEAK\n'
        '  :Units               hrs           hrs\n'
    )

    for i in range(len(subids)):
        subid = subids[i]
        time_conc = round(timeofconc[i], 2)
        time_peak = round(timetopeak[i], 2)

        file.write(f"{subid:>15}{time_conc:>10}{time_peak:>14}\n")

    file.write(':EndSubBasinProperties\n')

### 2.9 Create channel properties RVP file

In [81]:
# Create file and store in processed folder
FILE_PATH_RVP_CHANNEL = RA_PROCESSED_PATH + "channel_properties.rvp"


# Write to file
with open(FILE_PATH_RVP_CHANNEL, 'w') as file:
    # Write header content to file
    header = (
        "# ----------------------------------------------\n"
        f"# Raven Input file\n"
        f"# {WATERSHED_NAME}\n"
        "# ----------------------------------------------\n"
    )
    file.write(header)


# Append to file
with open(FILE_PATH_RVP_CHANNEL, 'a') as file:
    # Append channel transect data for each subbasin to file
    for i in range(len(subids)):
        # Define survey point x values based on bankfull depth and width
        x1 = 0
        x2 = bkfdepth_mod[40] * 2
        x3 = (bkfwidth[40] - (bkfdepth_mod[40] * 3)) * 2
        x4 = x2 + x3
        x5 = (bkfwidth[40] - (bkfdepth_mod[40] * 2)) + x3
        x6 = x5 + x2
        x7 = x3 - x2 + x6
        x8 = x7 + x2

        # Prepare formatted multiline string for this subbasin
        content = (
            f"\n###################################chn_{subids[i]}"
            f"\n## Subbasin ID:        {subids[i]}"
            f"\n## Subbasin area:      {round(basarea_km2[i], 2)} km2"
            f"\n## Channel length:     {round(rivlengths_mod_n[i], 2)} km"
            f"\n## Channel elevation:  {round(elev[i], 3)} m"
            f"\n## Bankfull width:     {round(bkfwidth[i], 4)} m"
            f"\n## Bankfull depth:     {round(bkfdepth_mod[i], 2)} m"
            f"\n:ChannelProfile       chn_{subids[i]}"
            f"\n    :Bedslope         {round(rivslope_mod[i], 7)}"
            f"\n    :SurveyPoints"
            f"\n        {x1: >5}  {round(elev[i] + bkfdepth_mod[i] + (bkfdepth_mod[i] / 2), 2)}"
            f"\n        {round(x2, 2): >5}  {round(elev[i] + bkfdepth_mod[i], 2)}"
            f"\n        {round(x3, 2): >5}  {round(elev[i] + bkfdepth_mod[i], 2)}"
            f"\n        {round(x4, 2): >5}  {round(elev[i], 2)}"
            f"\n        {round(x5, 2): >5}  {round(elev[i], 2)}"
            f"\n        {round(x6, 2): >5}  {round(elev[i] + bkfdepth_mod[i], 2)}"
            f"\n        {round(x7, 2): >5}  {round(elev[i] + bkfdepth_mod[i], 2)}"
            f"\n        {round(x8, 2): >5}  {round(elev[i] + bkfdepth_mod[i] + (bkfdepth_mod[0] / 2), 2)}"
            f"\n    :EndSurveyPoints"
            f"\n    :RoughnessZones"
            f"\n        {0: >5} {round(floodpn_mod[i], 6)}"
            f"\n        {round(x3, 2): >5} {round(chn_mod[i], 6)}"
            f"\n        {round(x6, 2): >5} {round(floodpn_mod[i], 6)}"
            f"\n    :EndRoughnessZones"
            f"\n:EndChannelProfile\n"
        )

        # Write the content to the file
        file.write(content)


### 2.10 Create GridWeights TXT file

In [82]:
# Create file and store in processed folder
FILE_PATH_GRIDWEIGHTS = RA_PROCESSED_PATH + "GridWeights.txt"


# Write to file
with open(FILE_PATH_GRIDWEIGHTS, 'w') as file:
    # Write header and GridWeights metadata to file
    file.write(
        f"# ----------------------------------------------\n"
        f"# Raven Input file\n"
        f"# {WATERSHED_NAME}\n"
        f"# ----------------------------------------------\n\n"
        f":GridWeights\n"
        f"   #\n"
        f"   # [# HRUs]\n"
        f"   :NumberHRUs       {len(subids)}\n"
        f"   :NumberGridCells  4175\n"
        f"   #\n"
        f"   # [HRU ID] [Cell #] [w_kl]"
    )


# Open file in append mode
with open(FILE_PATH_GRIDWEIGHTS, 'a') as file:
    # Append grid weights data to file
    for subid in subids:
        file.write(f"\n  {subid:>10}{108:>10}{1.0:>10}")
    file.write("\n:EndGridWeights")